# MATH-500 Dataset Explorer

Use this notebook to understand the 500 problems before running model experiments. It loads the label-free question view first and joins evaluation labels only for analysis.

> **Label firewall:** reference answers and solutions are for exploration and final evaluation only. Do not feed them into raw generation, synthesis, rewards, prompt selection, checkpoint selection, or RL training.

## 1. Setup

From the repository root, prepare the data and notebook environment if needed:

```bash
python3 scripts/prepare_math500.py
uv sync --extra notebook --frozen
uv run --extra notebook --frozen jupyter lab notebooks/math500_explorer.ipynb
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
plt.style.use("ggplot")

In [ ]:
def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs/datasets/math500.lock.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root. Launch Jupyter from this repository."
    )


REPOSITORY_ROOT = find_repository_root()
QUESTIONS_PATH = REPOSITORY_ROOT / "data/math500/questions.jsonl"
LABELS_PATH = REPOSITORY_ROOT / "data/math500/labels.jsonl"

missing = [path for path in (QUESTIONS_PATH, LABELS_PATH) if not path.is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing prepared data: {missing}. Run python3 scripts/prepare_math500.py first."
    )

print(f"Repository: {REPOSITORY_ROOT}")
print(f"Questions:  {QUESTIONS_PATH}")
print(f"Labels:     {LABELS_PATH}")

## 2. Load and validate

The assertions below make accidental schema drift or a broken question/label join visible immediately.

In [ ]:
questions = pd.read_json(QUESTIONS_PATH, lines=True)
labels = pd.read_json(LABELS_PATH, lines=True)

expected_question_columns = {"id", "problem"}
expected_label_columns = {"id", "answer", "solution", "subject", "level"}

assert set(questions.columns) == expected_question_columns
assert set(labels.columns) == expected_label_columns
assert len(questions) == len(labels) == 500
assert questions["id"].is_unique and labels["id"].is_unique
assert set(questions["id"]) == set(labels["id"])

dataset = questions.merge(labels, on="id", how="inner", validate="one_to_one", sort=False)
assert len(dataset) == 500

print(f"Loaded {len(questions)} questions and {len(labels)} label records.")
display(questions.head(5))

## 3. Dataset overview

In [ ]:
overview = pd.DataFrame(
    {
        "metric": [
            "Problems",
            "Subjects",
            "Difficulty levels",
            "Problems with [asy] diagram source",
            "Unique reference answers",
        ],
        "value": [
            len(dataset),
            dataset["subject"].nunique(),
            dataset["level"].nunique(),
            dataset["problem"].str.contains("[asy]", regex=False).sum(),
            dataset["answer"].nunique(),
        ],
    }
)
display(overview)

In [ ]:
subject_counts = dataset["subject"].value_counts().sort_values()
level_counts = dataset["level"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
subject_counts.plot.barh(ax=axes[0], color="#4C78A8")
axes[0].set_title("Problems by subject")
axes[0].set_xlabel("Problem count")
axes[0].set_ylabel("")

level_counts.plot.bar(ax=axes[1], color="#F58518", rot=0)
axes[1].set_title("Problems by difficulty level")
axes[1].set_xlabel("Level")
axes[1].set_ylabel("Problem count")

fig.tight_layout()
plt.show()

In [ ]:
subject_by_level = pd.crosstab(dataset["subject"], dataset["level"])
subject_by_level.columns = [f"Level {level}" for level in subject_by_level.columns]
display(subject_by_level)

## 4. Search and filter

Search results omit answers and solutions by default. Set `include_references=True` only when you intentionally want to inspect them.

In [ ]:
def search_problems(
    query: str = "",
    *,
    subject: str | None = None,
    level: int | None = None,
    limit: int = 10,
    include_references: bool = False,
) -> pd.DataFrame:
    result = dataset
    if query:
        result = result[result["problem"].str.contains(query, case=False, regex=False)]
    if subject is not None:
        result = result[result["subject"] == subject]
    if level is not None:
        result = result[result["level"] == level]

    columns = ["id", "subject", "level", "problem"]
    if include_references:
        columns.extend(["answer", "solution"])
    return result.loc[:, columns].head(limit).reset_index(drop=True)


search_problems("triangle", limit=5)

## 5. Sample individual problems

Choose a random problem, optionally restricted by subject and difficulty. The display hides the reference by default.

In [ ]:
def sample_problem(
    *, subject: str | None = None, level: int | None = None, seed: int | None = None
) -> pd.Series:
    candidates = dataset
    if subject is not None:
        candidates = candidates[candidates["subject"] == subject]
    if level is not None:
        candidates = candidates[candidates["level"] == level]
    if candidates.empty:
        raise ValueError("No problems match those filters.")
    return candidates.sample(n=1, random_state=seed).iloc[0]


def show_problem(record: pd.Series, *, reveal_reference: bool = False) -> None:
    display(Markdown(f"### {record['subject']} · Level {record['level']}"))
    display(Markdown(str(record["problem"])))
    print(f"ID: {record['id']}")
    if reveal_reference:
        display(Markdown("#### Reference answer"))
        display(Markdown(str(record["answer"])))
        display(Markdown("#### Reference solution"))
        display(Markdown(str(record["solution"])))


selected_problem = sample_problem(seed=17)
show_problem(selected_problem)

### Optional: reveal the selected reference

Change the flag to `True` only when you intend to inspect the evaluation label. A normal **Run All** keeps it hidden.

In [ ]:
REVEAL_REFERENCE = False

if REVEAL_REFERENCE:
    show_problem(selected_problem, reveal_reference=True)
else:
    print("Reference hidden. Set REVEAL_REFERENCE = True to inspect it.")

## 6. Inspect diagram-source problems

MATH-500 contains Asymptote source inside some problem statements. The dataset does not include rendered image files.

In [ ]:
diagram_problems = dataset[dataset["problem"].str.contains("[asy]", regex=False)]
print(f"Problems containing Asymptote source: {len(diagram_problems)}")
display(diagram_problems[["id", "subject", "level", "problem"]].head(5))

## Useful questions to explore next

- Which subjects and difficulty levels are weakest for the raw model?
- Does synthesis help uniformly, or mainly on levels 4–5?
- Do problems containing Asymptote source behave differently?
- How often does synthesis improve, preserve, or damage a correct raw rollout?

Keep official answers out of every training-stage table. Join them only in evaluation outputs.